# Segmenter Head Demo

Demonstrates DINOv3 segmentation using the `dinov3_vit7b16_ms` model with a Mask2Former head on an example image.

In [ ]:
import sys
from functools import partial

import matplotlib.pyplot as plt
import torch
from matplotlib import colormaps
from PIL import Image
from torchvision.transforms import v2

REPO_DIR = "../models/dinov3/"
sys.path.append(REPO_DIR)

from dinov3.eval.segmentation.inference import make_inference

## Helper Functions

In [ ]:
def get_img():
    import requests
    url = "http://images.cocodataset.org/val2017/000000039769.jpg"
    image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    return image


def make_transform(resize_size: int | list[int] = 768,
                   use_fp16: bool = False):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)

    dtype = torch.float16 if use_fp16 else torch.float32
    to_float = v2.ToDtype(dtype, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])

## Load Segmentor Model

Set `weights` to the segmentor checkpoint path/URL and `backbone_weights` to the DINOv3 backbone checkpoint path/URL.

In [ ]:
# Enable memory-efficient loading
torch.backends.mps.enabled = True  # For Apple Silicon
# torch.set_default_dtype(torch.float16)  # Use FP16 by default

In [ ]:
segmentor = torch.hub.load(
    REPO_DIR,
    "dinov3_vit7b16_ms",
    source="local",
    weights="../model_weights/dinov3_vit7b16_ade20k_m2f_head-bf307cb1.pth",
    # weights="https://dinov3.llamameta.net/dinov3_vit7b16/dinov3_vit7b16_ade20k_m2f_head-bf307cb1.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoibG05bWQxZnpiNnJ5anFzOTVndHQyODd2IiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NzIyODIwNzJ9fX1dfQ__&Signature=NgcDWH3nuJfGbQor7ZdNc0Czt0IQ76cj57YoxTwxw7qGml1H6eczboz9bOAGCJkMyxjWmnTsz7epd3FhDVAK8fUElRIM04pMsaCLPCEzBDCud9raTFPSNAVaeQkTMjfs1DZ7nVJh9JpRmmxCjQw9I50EsMwarNs-7RlpwzUuRsvpPY%7EbdP-pzYIQKytaZ2vip0OgUyf0lelj5udX8Qgvz0fuzJjtgY3GnmgsHOPQb5Lc7KADKOMl56GVnlu5KVeXGNqdOIrDZUNJJomVrRZNX1AiXIQnWSMyTYgqupQx2kBrcAQIqliCwD8amhH4tjf%7EbWB9JXv7lv8XnlpM8WivEw__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1561398151608429",
    backbone_weights="../model_weights/dinov3_vit7b16_pretrain_lvd1689m-a955f4ea.pth",
)

# segmentor = segmentor.half()

# If using MPS (Apple Silicon)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
segmentor = segmentor.to(device)

## Run Inference

In [ ]:
img_size = 896
img = get_img()
transform = make_transform(img_size, use_fp16=False)

with torch.inference_mode():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        batch_img = transform(img)[None].to(device)
        pred_vit7b = segmentor(batch_img)  # raw predictions
        # actual segmentation map
        segmentation_map_vit7b = make_inference(
            batch_img,
            segmentor,
            inference_mode="slide",
            decoder_head_type="m2f",
            rescale_to=(img.size[-1], img.size[-2]),
            n_output_channels=150,
            crop_size=(img_size, img_size),
            stride=(img_size, img_size),
            output_activation=partial(torch.nn.functional.softmax, dim=1),
        ).argmax(dim=1, keepdim=True)

## Visualize Results

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(121)
plt.imshow(img)
plt.axis("off")
plt.subplot(122)
plt.imshow(segmentation_map_vit7b[0, 0].cpu(), cmap=colormaps["Spectral"])
plt.axis("off")
plt.show()